In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 3.0.6
NumPy: 2.5.3


In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = None

    for path in [CURRENT_DIR, *CURRENT_DIR.parents]:
        if (path / "data" / "raw").is_dir() and (path / "notebooks").is_dir():
            PROJECT_ROOT = path
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f"Could not locate H-IDS project root from: {CURRENT_DIR}"
        )

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:")
print(PROJECT_ROOT)

print("\nRaw data:")
print(RAW_DATA_DIR)

print("\nProcessed data:")
print(PROCESSED_DATA_DIR)

Project root:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS

Raw data:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\raw

Processed data:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\processed


In [3]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

if len(csv_files) == 0:
    raise FileNotFoundError("No CSV files found in data/raw.")

print(f"Raw CSV files found: {len(csv_files)}")

for i, file in enumerate(csv_files, start=1):
    print(f"{i}. {file.name}")

Raw CSV files found: 8
1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
3. Friday-WorkingHours-Morning.pcap_ISCX.csv
4. Monday-WorkingHours.pcap_ISCX.csv
5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
7. Tuesday-WorkingHours.pcap_ISCX.csv
8. Wednesday-workingHours.pcap_ISCX.csv


In [4]:
# Load only the first raw CSV for a cleaning test

test_file = csv_files[0]

test_df = pd.read_csv(test_file)

print("File:")
print(test_file.name)

print("\nOriginal shape:")
print(test_df.shape)

print("\nOriginal label column:")
print(repr(test_df.columns[-1]))

File:
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv

Original shape:
(225745, 79)

Original label column:
' Label'


In [5]:
# Normalize column names

test_df.columns = test_df.columns.str.strip()

print("Last column after normalization:")
print(repr(test_df.columns[-1]))

Last column after normalization:
'Label'


In [6]:
# Normalize label text

label_mapping = {
    "Web Attack � Brute Force": "Web Attack - Brute Force",
    "Web Attack � XSS": "Web Attack - XSS",
    "Web Attack � Sql Injection": "Web Attack - Sql Injection",
}

test_df["Label"] = test_df["Label"].str.strip()

test_df["Label"] = test_df["Label"].replace(label_mapping)

print("Unique labels after normalization:")

for label in sorted(test_df["Label"].unique()):
    print(repr(label))

Unique labels after normalization:
'BENIGN'
'DDoS'


In [7]:
feature_columns = [
    column for column in test_df.columns
    if column != "Label"
]

print(f"Number of feature columns: {len(feature_columns)}")
print(f"Number of label columns: {test_df['Label'].nunique()}")

Number of feature columns: 78
Number of label columns: 2


In [8]:
# Convert all 78 network-flow features to numeric

for column in feature_columns:
    test_df[column] = pd.to_numeric(
        test_df[column],
        errors="coerce"
    )

print("Feature conversion completed.")

Feature conversion completed.


In [9]:
# Replace positive and negative infinity with NaN

test_df[feature_columns] = test_df[feature_columns].replace(
    [np.inf, -np.inf],
    np.nan
)

missing_before_drop = int(
    test_df[feature_columns].isna().sum().sum()
)

rows_before_drop = len(test_df)

test_df = test_df.dropna(
    subset=feature_columns
).reset_index(drop=True)

rows_after_drop = len(test_df)

rows_removed = rows_before_drop - rows_after_drop

print(f"Missing/infinite feature cells detected: {missing_before_drop:,}")
print(f"Rows before removal: {rows_before_drop:,}")
print(f"Rows removed: {rows_removed:,}")
print(f"Rows remaining: {rows_after_drop:,}")

Missing/infinite feature cells detected: 68
Rows before removal: 225,745
Rows removed: 34
Rows remaining: 225,711


In [10]:
# Validate the cleaned test data

remaining_missing = int(
    test_df[feature_columns].isna().sum().sum()
)

remaining_infinite = int(
    np.isinf(test_df[feature_columns].to_numpy()).sum()
)

remaining_non_numeric = test_df[feature_columns].select_dtypes(
    exclude="number"
).shape[1]

print("CLEANING VALIDATION")
print("=" * 50)

print(f"Rows: {len(test_df):,}")
print(f"Columns: {len(test_df.columns)}")
print(f"Features: {len(feature_columns)}")
print(f"Missing feature values: {remaining_missing}")
print(f"Infinite feature values: {remaining_infinite}")
print(f"Non-numeric feature columns: {remaining_non_numeric}")
print(f"Missing labels: {test_df['Label'].isna().sum()}")

CLEANING VALIDATION
Rows: 225,711
Columns: 79
Features: 78
Missing feature values: 0
Infinite feature values: 0
Non-numeric feature columns: 0
Missing labels: 0


In [11]:
# Clean all CIC-IDS2017 CSV files

label_mapping = {
    "Web Attack � Brute Force": "Web Attack - Brute Force",
    "Web Attack � XSS": "Web Attack - XSS",
    "Web Attack � Sql Injection": "Web Attack - Sql Injection",
}

cleaned_dataframes = {}
cleaning_summary = []

for file in csv_files:

    print(f"\nProcessing: {file.name}")

    # Load raw file
    df = pd.read_csv(file)

    rows_before = len(df)

    # 1. Normalize column names
    df.columns = df.columns.str.strip()

    # 2. Normalize labels
    df["Label"] = df["Label"].astype(str).str.strip()
    df["Label"] = df["Label"].replace(label_mapping)

    # 3. Identify feature columns
    feature_columns = [
        column for column in df.columns
        if column != "Label"
    ]

    # 4. Convert features to numeric
    for column in feature_columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

    # 5. Replace infinite values with NaN
    df[feature_columns] = df[feature_columns].replace(
        [np.inf, -np.inf],
        np.nan
    )

    # 6. Count invalid rows before removal
    invalid_rows = int(
        df[feature_columns].isna().any(axis=1).sum()
    )

    # 7. Remove rows with invalid feature values
    df = df.dropna(
        subset=feature_columns
    ).reset_index(drop=True)

    rows_after = len(df)

    # Store cleaned dataframe
    cleaned_dataframes[file.name] = df

    # Store cleaning statistics
    cleaning_summary.append({
        "file": file.name,
        "rows_before": rows_before,
        "rows_removed": rows_before - rows_after,
        "rows_after": rows_after,
        "feature_count": len(feature_columns),
        "label_count": df["Label"].nunique()
    })

    print(f"  Rows before : {rows_before:,}")
    print(f"  Rows removed: {rows_before - rows_after:,}")
    print(f"  Rows after  : {rows_after:,}")

cleaning_summary_df = pd.DataFrame(cleaning_summary)

print("\nAll files cleaned successfully.")


Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Rows before : 225,745
  Rows removed: 34
  Rows after  : 225,711

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Rows before : 286,467
  Rows removed: 371
  Rows after  : 286,096

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Rows before : 191,033
  Rows removed: 122
  Rows after  : 190,911

Processing: Monday-WorkingHours.pcap_ISCX.csv
  Rows before : 529,918
  Rows removed: 437
  Rows after  : 529,481

Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Rows before : 288,602
  Rows removed: 207
  Rows after  : 288,395

Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Rows before : 170,366
  Rows removed: 135
  Rows after  : 170,231

Processing: Tuesday-WorkingHours.pcap_ISCX.csv
  Rows before : 445,909
  Rows removed: 264
  Rows after  : 445,645

Processing: Wednesday-workingHours.pcap_ISCX.csv
  Rows before : 692,703
  Rows removed: 1,297
  Rows aft

In [12]:
print("TOTAL CLEANING SUMMARY")
print("=" * 60)

print(
    f"Rows before cleaning: "
    f"{cleaning_summary_df['rows_before'].sum():,}"
)

print(
    f"Rows removed: "
    f"{cleaning_summary_df['rows_removed'].sum():,}"
)

print(
    f"Rows after cleaning: "
    f"{cleaning_summary_df['rows_after'].sum():,}"
)

TOTAL CLEANING SUMMARY
Rows before cleaning: 2,830,743
Rows removed: 2,867
Rows after cleaning: 2,827,876


In [13]:
# Validate all cleaned DataFrames

validation_results = []

for file_name, df in cleaned_dataframes.items():

    feature_columns = [
        column for column in df.columns
        if column != "Label"
    ]

    missing_values = int(
        df[feature_columns].isna().sum().sum()
    )

    infinite_values = int(
        np.isinf(df[feature_columns].to_numpy()).sum()
    )

    non_numeric_columns = len(
        df[feature_columns].select_dtypes(exclude="number").columns
    )

    missing_labels = int(
        df["Label"].isna().sum()
    )

    validation_results.append({
        "file": file_name,
        "rows": len(df),
        "features": len(feature_columns),
        "missing_values": missing_values,
        "infinite_values": infinite_values,
        "non_numeric_features": non_numeric_columns,
        "missing_labels": missing_labels
    })

validation_df = pd.DataFrame(validation_results)

validation_df

,file,rows,features,missing_values,infinite_values,non_numeric_features,missing_labels
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225711,78,0,0,0,0
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286096,78,0,0,0,0
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,190911,78,0,0,0,0
3,Monday-WorkingHours.pcap_ISCX.csv,529481,78,0,0,0,0
4,Thursday-WorkingHours-Afternoon-Infilteration....,288395,78,0,0,0,0
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170231,78,0,0,0,0
6,Tuesday-WorkingHours.pcap_ISCX.csv,445645,78,0,0,0,0
7,Wednesday-workingHours.pcap_ISCX.csv,691406,78,0,0,0,0


In [14]:
print("CLEANED DATA VALIDATION")
print("=" * 60)

print(
    "Total missing feature values:",
    validation_df["missing_values"].sum()
)

print(
    "Total infinite feature values:",
    validation_df["infinite_values"].sum()
)

print(
    "Total non-numeric feature columns:",
    validation_df["non_numeric_features"].sum()
)

print(
    "Total missing labels:",
    validation_df["missing_labels"].sum()
)

CLEANED DATA VALIDATION
Total missing feature values: 0
Total infinite feature values: 0
Total non-numeric feature columns: 0
Total missing labels: 0


In [15]:
# Save individually cleaned datasets

for file_name, df in cleaned_dataframes.items():
    
    output_path = PROCESSED_DATA_DIR / file_name
    
    df.to_csv(
        output_path,
        index=False
    )
    
    print(f"Saved: {output_path.name}")

Saved: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Saved: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Saved: Friday-WorkingHours-Morning.pcap_ISCX.csv
Saved: Monday-WorkingHours.pcap_ISCX.csv
Saved: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Saved: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Saved: Tuesday-WorkingHours.pcap_ISCX.csv
Saved: Wednesday-workingHours.pcap_ISCX.csv


In [16]:
processed_files = sorted(PROCESSED_DATA_DIR.glob("*.csv"))

print(f"Processed CSV files: {len(processed_files)}")

for file in processed_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name}: {size_mb:.2f} MB")

Processed CSV files: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 78.70 MB
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 82.45 MB
Friday-WorkingHours-Morning.pcap_ISCX.csv: 60.84 MB
Monday-WorkingHours.pcap_ISCX.csv: 182.92 MB
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 87.64 MB
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 54.28 MB
Tuesday-WorkingHours.pcap_ISCX.csv: 141.14 MB
Wednesday-workingHours.pcap_ISCX.csv: 232.84 MB


In [17]:
# Load cleaned files and calculate exact row hashes

processed_files = sorted(PROCESSED_DATA_DIR.glob("*.csv"))

cleaned_row_hashes = {}

for file in processed_files:
    print(f"Hashing: {file.name}")

    df = pd.read_csv(file)

    hashes = pd.util.hash_pandas_object(
        df,
        index=False
    )

    cleaned_row_hashes[file.name] = hashes

    print(f"  Rows: {len(df):,}")

print("\nHashing completed.")

Hashing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Rows: 225,711
Hashing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Rows: 286,096
Hashing: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Rows: 190,911
Hashing: Monday-WorkingHours.pcap_ISCX.csv
  Rows: 529,481
Hashing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Rows: 288,395
Hashing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Rows: 170,231
Hashing: Tuesday-WorkingHours.pcap_ISCX.csv
  Rows: 445,645
Hashing: Wednesday-workingHours.pcap_ISCX.csv
  Rows: 691,406

Hashing completed.


In [18]:
# Count exact rows that occur across multiple processed files

hash_to_files_cleaned = {}

for file_name, hashes in cleaned_row_hashes.items():

    for row_hash in hashes:
        hash_to_files_cleaned.setdefault(
            row_hash,
            set()
        ).add(file_name)

cross_file_cleaned = {
    row_hash: files
    for row_hash, files in hash_to_files_cleaned.items()
    if len(files) > 1
}

print(
    "Unique exact rows appearing in multiple cleaned files:",
    len(cross_file_cleaned)
)

Unique exact rows appearing in multiple cleaned files: 33530


In [19]:
# Combine all cleaned datasets

cleaned_dfs = []

for file in processed_files:
    print(f"Loading: {file.name}")
    
    df = pd.read_csv(file)
    cleaned_dfs.append(df)

combined_cleaned_df = pd.concat(
    cleaned_dfs,
    ignore_index=True
)

print("\nCombined cleaned dataset:")
print(f"Rows: {len(combined_cleaned_df):,}")
print(f"Columns: {len(combined_cleaned_df.columns)}")

Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
Loading: Monday-WorkingHours.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Loading: Tuesday-WorkingHours.pcap_ISCX.csv
Loading: Wednesday-workingHours.pcap_ISCX.csv

Combined cleaned dataset:
Rows: 2,827,876
Columns: 79


In [20]:
# Measure global exact duplicates

global_duplicate_count = int(
    combined_cleaned_df.duplicated().sum()
)

unique_rows = (
    len(combined_cleaned_df) - global_duplicate_count
)

print("GLOBAL DUPLICATE ANALYSIS")
print("=" * 60)

print(f"Total cleaned rows : {len(combined_cleaned_df):,}")
print(f"Duplicate rows     : {global_duplicate_count:,}")
print(f"Unique rows        : {unique_rows:,}")

GLOBAL DUPLICATE ANALYSIS
Total cleaned rows : 2,827,876
Duplicate rows     : 307,078
Unique rows        : 2,520,798


In [21]:
# Class distribution before duplicate removal

label_distribution_before = (
    combined_cleaned_df["Label"]
    .value_counts()
    .rename("count")
    .to_frame()
)

label_distribution_before["percentage"] = (
    label_distribution_before["count"]
    / len(combined_cleaned_df)
    * 100
)

label_distribution_before

,count,percentage
Label,,
BENIGN,2271320,80.318939
DoS Hulk,230124,8.137698
PortScan,158804,5.615663
DDoS,128025,4.527249
DoS GoldenEye,10293,0.363983
FTP-Patator,7935,0.280599
SSH-Patator,5897,0.208531
DoS slowloris,5796,0.204959
DoS Slowhttptest,5499,0.194457


In [22]:
# Create a temporary duplicate-free dataset

deduplicated_df = combined_cleaned_df.drop_duplicates(
    ignore_index=True
)

print("Rows before duplicate removal:",
      f"{len(combined_cleaned_df):,}")

print("Rows after duplicate removal:",
      f"{len(deduplicated_df):,}")

print("Rows removed:",
      f"{len(combined_cleaned_df) - len(deduplicated_df):,}")

Rows before duplicate removal: 2,827,876
Rows after duplicate removal: 2,520,798
Rows removed: 307,078


In [23]:
# Class distribution after duplicate removal

label_distribution_after = (
    deduplicated_df["Label"]
    .value_counts()
    .rename("count_after")
    .to_frame()
)

comparison = label_distribution_before.join(
    label_distribution_after
)

comparison["removed"] = (
    comparison["count"] -
    comparison["count_after"]
)

comparison["percentage_after"] = (
    comparison["count_after"]
    / len(deduplicated_df)
    * 100
)

comparison

,count,percentage,count_after,removed,percentage_after
Label,,,,,
BENIGN,2271320,80.318939,2095057,176263,83.110864
DoS Hulk,230124,8.137698,172846,57278,6.856797
PortScan,158804,5.615663,90694,68110,3.597829
DDoS,128025,4.527249,128014,11,5.078313
DoS GoldenEye,10293,0.363983,10286,7,0.408045
FTP-Patator,7935,0.280599,5931,2004,0.235283
SSH-Patator,5897,0.208531,3219,2678,0.127698
DoS slowloris,5796,0.204959,5385,411,0.213623
DoS Slowhttptest,5499,0.194457,5228,271,0.207395


In [24]:
# Save the final cleaned and deduplicated dataset

FINAL_DATA_PATH = (
    PROCESSED_DATA_DIR /
    "cic_ids2017_cleaned_deduplicated.parquet"
)

deduplicated_df.to_parquet(
    FINAL_DATA_PATH,
    index=False
)

print("Final processed dataset saved:")
print(FINAL_DATA_PATH)

print("\nShape:")
print(deduplicated_df.shape)

Final processed dataset saved:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\processed\cic_ids2017_cleaned_deduplicated.parquet

Shape:
(2520798, 79)


In [28]:
print("Parquet file exists:", FINAL_DATA_PATH.exists())

if FINAL_DATA_PATH.exists():
    size_mb = FINAL_DATA_PATH.stat().st_size / (1024 ** 2)
    print(f"Parquet file size: {size_mb:.2f} MB")

Parquet file exists: True
Parquet file size: 305.13 MB


In [30]:
import gc

for variable_name in [
    "dataframes",
    "cleaned_dataframes",
    "cleaned_dfs",
    "combined_cleaned_df"
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

print("Unused large DataFrames released from memory.")

Unused large DataFrames released from memory.


In [31]:
print("deduplicated_df exists:", "deduplicated_df" in globals())

if "deduplicated_df" in globals():
    print(f"Final rows: {len(deduplicated_df):,}")
    print(f"Final columns: {len(deduplicated_df.columns)}")

deduplicated_df exists: True
Final rows: 2,520,798
Final columns: 79


In [32]:
final_label_distribution = (
    deduplicated_df["Label"]
    .value_counts()
    .rename_axis("Label")
    .reset_index(name="Count")
)

final_label_distribution["Percentage"] = (
    final_label_distribution["Count"]
    / len(deduplicated_df)
    * 100
)

final_label_distribution

,Label,Count,Percentage
0,BENIGN,2095057,83.110864
1,DoS Hulk,172846,6.856797
2,DDoS,128014,5.078313
3,PortScan,90694,3.597829
4,DoS GoldenEye,10286,0.408045
5,FTP-Patator,5931,0.235283
6,DoS slowloris,5385,0.213623
7,DoS Slowhttptest,5228,0.207395
8,SSH-Patator,3219,0.127698
9,Bot,1948,0.077277


In [33]:
FINAL_LABEL_REPORT_PATH = (
    PROCESSED_DATA_DIR / "final_label_distribution.csv"
)

final_label_distribution.to_csv(
    FINAL_LABEL_REPORT_PATH,
    index=False
)

print("Label distribution saved to:")
print(FINAL_LABEL_REPORT_PATH)

Label distribution saved to:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\processed\final_label_distribution.csv


In [34]:
CLEANING_SUMMARY_PATH = (
    PROCESSED_DATA_DIR / "cleaning_summary.csv"
)

cleaning_summary_df.to_csv(
    CLEANING_SUMMARY_PATH,
    index=False
)

print("Cleaning summary saved to:")
print(CLEANING_SUMMARY_PATH)

Cleaning summary saved to:
c:\Users\LENOVO\Desktop\Hybrid IDS\H-IDS\data\processed\cleaning_summary.csv
